In [6]:
import os
import sys
import csv
import wave
import copy
import math

import numpy as np
import pandas as pd

from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.svm import OneClassSVM, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from keras.models import Sequential, Model
from keras.layers import Dense, Activation, LSTM, Input, TimeDistributed
from keras.layers import LSTM, Input
from keras.layers import Dense, Activation, LSTM, Input, TimeDistributed
from keras.optimizers import SGD, Adam, RMSprop

sys.path.append("../")

#from utilities.utils import *

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

from IPython.display import clear_output

In [7]:
batch_size = 64
nb_feat = 34
nb_class = 4
nb_epoch = 80

optimizer = 'Adadelta'

In [15]:
import pickle

data_path = 'C:/Users/91984/Desktop/project work/data_collected_full.pickle'

with open(data_path, 'rb') as handle:
    data2 = pickle.load(handle)


In [20]:
code_path = os.path.dirname(os.path.realpath(os.getcwd()))
emotions_used = np.array(['ang', 'exc', 'neu', 'sad'])
data_path = code_path + "/../data/sessions/"
sessions = ['Session1', 'Session2', 'Session3', 'Session4', 'Session5']
framerate = 16000

In [23]:
from features import *
from helper import *

In [25]:
len(data2)

4936

In [27]:
data2[2]

{'start': 14.8872,
 'end': 18.0175,
 'id': 'Ses01F_impro01_F002',
 'v': 2.5,
 'a': 2.5,
 'd': 2.5,
 'emotion': 'neu',
 'emo_evo': [['neu'], ['sur'], ['neu'], ['neu', 'ang']],
 'signal': array([-12,  -5,  14, ...,  -6,  29,  41], dtype=int16),
 'transcription': 'Is there a problem?',
 'mocap_hand': [array([[       nan,        nan,        nan,  162.46902, -110.20654,
           -37.91526,  259.065  , -181.07736,  -78.59225,        nan,
                 nan,        nan,   81.34849, -116.14007,  -13.03359,
           -15.96942, -199.91057,  -36.55597],
         [       nan,        nan,        nan,  162.44339, -110.24825,
           -37.89717,  259.04422, -181.11484,  -78.61979,        nan,
                 nan,        nan,   81.29459, -116.17288,  -13.05965,
           -16.04064, -199.92699,  -36.5386 ]]),
  array([[       nan,        nan,        nan,  162.42973, -110.28729,
           -37.90519,  258.99387, -181.14156,  -78.64119,        nan,
                 nan,        nan,   81.21877, 

In [31]:
import numpy as np

In [42]:
import numpy as np

# Initialize list to hold processed mocap data
x_train_mocap = []

# Iterate over each session in the data
for ses_mod in data2:
    # Initialize default shapes
    x_head = np.zeros((200, 18))  # Placeholder for head data
    x_hand = np.zeros((200, 6))    # Placeholder for hand data
    x_rot = np.zeros((200, 165))   # Placeholder for rotation data

    # Process mocap_head
    head_data = ses_mod['mocap_head']
    for i in range(min(len(head_data), 200)):  # Ensure we do not exceed bounds
        current_head = head_data[i].flatten()  # Flatten the array
        if len(current_head) >= 18:
            x_head[i] = current_head[:18]  # Truncate to fit
        else:
            x_head[i, :len(current_head)] = current_head  # Fill with actual data
            # Pad the rest with zeros
            x_head[i, len(current_head):] = 0  # Assuming you want to pad with zeros

    # Process mocap_hand
    hand_data = ses_mod['mocap_hand']
    for i in range(min(len(hand_data), 200)):  # Ensure we do not exceed bounds
        current_hand = hand_data[i].flatten()  # Flatten the array
        if len(current_hand) >= 6:
            x_hand[i] = current_hand[:6]  # Truncate to fit
        else:
            x_hand[i, :len(current_hand)] = current_hand  # Fill with actual data
            # Pad the rest with zeros
            x_hand[i, len(current_hand):] = 0  # Padding with zeros

    # Process mocap_rot
    rot_data = ses_mod['mocap_rot']
    for i in range(min(len(rot_data), 200)):  # Ensure we do not exceed bounds
        current_rot = rot_data[i].flatten()  # Flatten the array
        if len(current_rot) >= 165:
            x_rot[i] = np.nan_to_num(current_rot[:165], nan=0.0)  # Truncate to fit
        else:
            x_rot[i, :len(current_rot)] = np.nan_to_num(current_rot, nan=0.0)  # Fill with actual data
            # Pad the rest with zeros
            x_rot[i, len(current_rot):] = 0  # Padding with zeros

    # Concatenate the features
    x_mocap = np.concatenate((x_head, x_hand), axis=1)
    x_mocap = np.concatenate((x_mocap, x_rot), axis=1)
    x_train_mocap.append(x_mocap)

# Convert the list of mocap data to a NumPy array
x_train_mocap = np.array(x_train_mocap)

# Reshape the data for model input
x_train_mocap = x_train_mocap.reshape(-1, 200, 189, 1)

# Print the shape for verification
print(x_train_mocap.shape)  # Should print (4936, 200, 189, 1)


(4936, 200, 189, 1)


In [44]:
from sklearn.preprocessing import label_binarize

Y = []
for ses_mod in data2:
    Y.append(ses_mod['emotion'])

# Check what emotions_used is
print("Emotions used:", emotions_used)

# Use label_binarize correctly
Y = label_binarize(Y, classes=emotions_used)

# Check the shape of Y
print("Shape of y_train:", Y.shape)

Emotions used: ['ang' 'exc' 'neu' 'sad']
Shape of y_train: (4936, 4)


In [66]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

# Define constants
n = 34  # Number of segments
h = 112  # Height of each frame (reduced)
w = 112  # Width of each frame (reduced)
c = 3    # Number of channels (RGB)
nb_class = 4  # Number of output classes

def create_tsn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    
    # TSN Layer 1: Convolutional Layer
    x = layers.Conv3D(64, (3, 3, 3), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)

    # TSN Layer 2: Convolutional Layer
    x = layers.Conv3D(128, (3, 3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)

    # TSN Layer 3: Convolutional Layer
    x = layers.Conv3D(256, (3, 3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling3D()(x)

    # Dense layer with fewer units
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # Output layer for classification
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    # Create the model
    model = Model(inputs=inputs, outputs=outputs)
    
    # Compile the model
    model.compile(loss='categorical_crossentropy', optimizer='Adadelta', metrics=['accuracy'])
    
    return model

# Specify input shape as (n, h, w, c)
input_shape = (n, h, w, c)

# Clear any previous session
tf.keras.backend.clear_session()

# Instantiate the TSN model
tsn_model = create_tsn_model(input_shape, nb_class)
tsn_model.summary()  # Print the model summary


ResourceExhaustedError: {{function_node __wrapped__StatelessRandomUniformV2_device_/job:localhost/replica:0/task:0/device:CPU:0}} OOM when allocating tensor with shape[3,3,3,128,256] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator mklcpu [Op:StatelessRandomUniformV2] name: 

In [64]:
import numpy as np
import cv2
import tensorflow as tf

# Assuming x_train_mocap is your original data and Y are your target labels
# x_train_mocap shape: (4936, 200, height, width, channels)
# Y shape: (4936, 4) for one-hot encoded labels (4 classes)

# Splitting the training data
xtrain_sp = x_train_mocap[:3838]  # Training data
xtest_sp = x_train_mocap[3838:]    # Testing data
ytrain_sp = Y[:3838]                # Training labels
ytest_sp = Y[3838:]                  # Testing labels

# Resizing data (if necessary) and converting to float32
n_frames = 2  # Number of frames for the model
h, w = 112, 112  # Height and width for resizing
xtrain_resized = np.zeros((len(xtrain_sp), n_frames, h, w, 3), dtype=np.float32)
xtest_resized = np.zeros((len(xtest_sp), n_frames, h, w, 3), dtype=np.float32)

for i in range(len(xtrain_sp)):
    for j in range(n_frames):
        frame_index = j % xtrain_sp.shape[1]  # Wrap around if not enough frames
        frame = xtrain_sp[i, frame_index]
        frame_resized = cv2.resize(frame, (w, h))
        if frame_resized.ndim == 2:  # Ensure frame is RGB
            frame_resized = cv2.merge([frame_resized] * 3)
        xtrain_resized[i, j] = frame_resized

for i in range(len(xtest_sp)):
    for j in range(n_frames):
        frame_index = j % xtest_sp.shape[1]  # Wrap around if not enough frames
        frame = xtest_sp[i, frame_index]
        frame_resized = cv2.resize(frame, (w, h))
        if frame_resized.ndim == 2:  # Ensure frame is RGB
            frame_resized = cv2.merge([frame_resized] * 3)
        xtest_resized[i, j] = frame_resized


# Fit the model using the training data
try:
    hist = tsn_model.fit(
        xtrain_resized,
        ytrain_sp,
        epochs=80,
        verbose=1,
        validation_data=(xtest_resized, ytest_sp),
        batch_size=34  # You can adjust the batch size as needed
    )
except Exception as e:
    print("An error occurred during training:", str(e))


Epoch 1/80
An error occurred during training: Graph execution error:

Detected at node adadelta/Mul_12 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "C:\Users\91984\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\91984\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "C:\Users\91984\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "C:\Users\91984\AppData\Roaming\Python\Python311\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "c:\Users\91984\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 604, in run_forever

  File "c:\Users\91984\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 1909, in _run_once

  File "c:\Users\91984\AppData\Lo